[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Peewee, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/peewee-deep-dive.html)

# JSON Columns &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's Setup, with the `Edition` model and the four editions. Run it
first. Task 4 changes the documents, so run these in order.


In [1]:
import json
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

try:
    if version("peewee") != "4.5.1":                                # Colab has 4.4.0, whose wording differs
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "peewee==4.5.1"], check=True)

import peewee
from peewee import (CharField, IntegerField, JSONField, Model, OperationalError, SqliteDatabase,
                    TextField)

def sql(query):
    """The SQL a query will send, and the values that go with it, on one line."""
    statement, values = query.sql()
    return " ".join(statement.split()) + (f"  {values}" if values else "")

EDITIONS = [                                                        # title, the edition as a document
    ("The Salt Road", {"format": "hardback", "pages": 312, "printing": 3,
                       "rights": {"regions": ["uk", "us"], "audio": True},
                       "tags": ["debut", "prize"]}),
    ("Nightjar", {"format": "paperback", "pages": 244, "printing": 11,
                  "rights": {"regions": ["uk"], "audio": False},
                  "tags": ["reprint"]}),
    ("Stone and Tide", {"format": "hardback", "pages": 501, "printing": 2,
                        "rights": {"regions": ["uk", "us", "ca"], "audio": True},
                        "tags": ["prize", "translated"]}),
    ("A Careful Fire", {"format": "paperback", "pages": 420, "printing": 9,
                        "rights": {"regions": ["ie", "uk"], "audio": False},
                        "tags": []}),
]

db = SqliteDatabase(":memory:", pragmas={"foreign_keys": 1})        # SQLite enforces nothing without this


class Edition(Model):
    """One row per edition, with everything that varies by edition in one JSON column."""

    title = CharField(max_length=80)
    detail = JSONField()

    class Meta:
        database = db


db.create_tables([Edition])
for title, detail in EDITIONS:
    Edition.create(title=title, detail=detail)

print("peewee", peewee.__version__, "|", Edition.select().count(), "editions |",
      "detail comes back as a", type(Edition.get().detail).__name__)


peewee 4.5.1 | 4 editions | detail comes back as a dict


**1.** Formats, read and then queried.


In [2]:
for row in Edition.select().order_by(Edition.title):
    print(f"  {row.title:<16} {row.detail['format']}")

paperbacks = Edition.select().where(Edition.detail["format"] == "paperback")
print()
print("paperbacks:", [row.title for row in paperbacks])


  A Careful Fire   paperback
  Nightjar         paperback
  Stone and Tide   hardback
  The Salt Road    hardback

paperbacks: ['Nightjar', 'A Careful Fire']


The first loop reads the document in Python, where it is a dict. The query asks the database
instead, which is what lets it be a filter rather than something done to every row after fetching
it.


**2.** The same threshold, with and without a cast.


In [3]:
without = Edition.select().where(Edition.detail["pages"] > 400)
with_cast = Edition.select().where(Edition.detail["pages"].as_int() > 400)

print("no cast:", [(row.title, row.detail["pages"]) for row in without])
print("cast:   ", [(row.title, row.detail["pages"]) for row in with_cast])
print()
print("no cast:", sql(without).split("WHERE ")[1])
print("cast:   ", sql(with_cast).split("WHERE ")[1])


no cast: [('Stone and Tide', 501), ('A Careful Fire', 420)]
cast:    [('Stone and Tide', 501), ('A Careful Fire', 420)]

no cast: (("t1"."detail" -> ?) > json(?))  ['$."pages"', '400']
cast:    (CAST(("t1"."detail" ->> ?) AS INTEGER) > ?)  ['$."pages"', 400]


The two answers agree, which is the uncomfortable part. Every page count here has three digits, so
comparing them as text happens to put them in the same order as comparing them as numbers. The
clauses show that the questions are different: one is `->` against `json(?)` and the other is a
`CAST` against a number. Change the data to a four digit page count and only one of them stays
right.


**3.** Printings, in order.


In [4]:
in_order = Edition.select().order_by(Edition.detail["printing"].as_int())

for row in in_order:
    print(f"  {row.detail['printing']:>3}  {row.title}")
print()
print("without the cast:", [row.detail["printing"] for row in
                            Edition.select().order_by(Edition.detail["printing"])])


    2  Stone and Tide
    3  The Salt Road
    9  A Careful Fire
   11  Nightjar

without the cast: [11, 2, 3, 9]


Eleven sorts before two as text, which is what the second line shows. The cast is needed in
`order_by` for the same reason it is needed in `where`.


**4.** A key added to every document, in one statement.


In [5]:
query = Edition.update({Edition.detail: Edition.detail.update({"price": 12.99})})
print(sql(query))
print("rows changed:", query.execute())

for row in Edition.select().order_by(Edition.title):
    print(f"  {row.title:<16} price {row.detail['price']} | printing {row.detail['printing']}")


UPDATE "edition" SET "detail" = json_patch("edition"."detail", json(?))  ['{"price": 12.99}']
rows changed: 4
  A Careful Fire   price 12.99 | printing 9
  Nightjar         price 12.99 | printing 11
  Stone and Tide   price 12.99 | printing 2
  The Salt Road    price 12.99 | printing 3


`json_patch` merged the new key into each document and left everything else alone. No document was
read into Python, so nothing that another writer had changed in the meantime could be overwritten.


**5.** Tag counts.


In [6]:
counted = Edition.select(Edition.title, Edition.detail["tags"].length().alias("tags"))

for row in counted.order_by(Edition.title):
    print(f"  {row.title:<16} {row.tags} tag{'' if row.tags == 1 else 's'}")

print()
print("no tags at all:", [row.title for row in
                          Edition.select().where(Edition.detail["tags"].length() == 0)])


  A Careful Fire   0 tags
  Nightjar         1 tag
  Stone and Tide   2 tags
  The Salt Road    2 tags

no tags at all: ['A Careful Fire']


`json_array_length` counts in the database, so the filter on it is a filter rather than a pass over
every row in Python.


**6.** The same documents in a `TextField`.


In [7]:
class Mistake(Model):
    title = CharField(max_length=80)
    detail = TextField()

    class Meta:
        database = db


db.create_tables([Mistake])
for title, detail in EDITIONS:
    Mistake.create(title=title, detail=json.dumps(detail))

row = Mistake.get(Mistake.title == "The Salt Road")
print("type:", type(row.detail).__name__)
try:
    row.detail["format"]
except TypeError as error:
    print("in Python:", type(error).__name__ + ":", error)

query = Mistake.select().where(Mistake.detail["format"] == "hardback")
print("in a query:", len(query), "rows, from", Mistake.select().count())
print("the clause:", sql(query).split("WHERE ")[1])


type: str
in Python: TypeError: string indices must be integers, not 'str'
in a query: 0 rows, from 4
the clause: (("t1"."detail" = ?) = ?)  ['format', 'hardback']


Python raises, which is the half you find. The query does not, and its `WHERE` clause never mentions
JSON: it compares the whole column with the string `format` and then compares that result with
`hardback`. One word in the model, `JSONField` instead of `TextField`, is the whole fix.


---

&#8592; **Back to:** [JSON Columns](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/peewee-deep-dive/08-json-columns.ipynb)  &nbsp;&middot;&nbsp;  [Peewee, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/peewee-deep-dive.html)
